In [19]:
#%pip install pyspark

In [20]:
# Importa a classe principal para criar e iniciar a sessão Spark
from pyspark.sql import SparkSession

# Importa funções do PySpark utilizadas para limpeza e transformação dos dados
from pyspark.sql.functions import col, trim, lower, regexp_replace, when

# Pandas será usado apenas na etapa final para exportar o DataFrame tratado para CSV
import pandas as pd

# Biblioteca para uso de expressões regulares
import re

# Biblioteca para remover acentos e caracteres especiais dos textos
import unicodedata

# Biblioteca usada para verificar se o arquivo final já existe e removê-lo antes de salvar
import os


# =========================
# 1. CRIAÇÃO DA SESSÃO SPARK
# =========================

# Inicializa a sessão Spark, responsável por permitir a leitura,
# transformação e manipulação dos dados em PySpark
spark = SparkSession.builder \
    .appName("WeatherForecastCleaning") \
    .getOrCreate()


# =========================
# 2. DEFINIÇÃO DOS CAMINHOS
# =========================

# Caminho do arquivo CSV original com os dados brutos da previsão do tempo
arquivo_entrada = r"C:\Users\gques\Documents\SPTECH_CODIGOS\ultimo-ano\tcc\previsao_15_dias.csv"

# Caminho do arquivo CSV final já tratado pelo processo ETL
arquivo_saida = r"C:\Users\gques\Documents\SPTECH_CODIGOS\ultimo-ano\tcc\weather_forecast_15_days_clean.csv"


# =========================
# 3. EXTRAÇÃO (E DO ETL)
# =========================

# Lê o arquivo CSV bruto
# header=True -> usa a primeira linha como nome das colunas
# inferSchema=False -> lê tudo como string para dar mais controle ao tratamento
df = spark.read.option("header", True).option("inferSchema", False).csv(arquivo_entrada)


# =========================
# 4. FUNÇÃO PARA PADRONIZAR TEXTO EM SNAKE_CASE
# =========================

# Essa função será usada para padronizar nomes de colunas
# e garantir que fiquem no formato:
# - minúsculo
# - sem acento
# - sem caracteres especiais
# - com espaços substituídos por underline (_)
def to_snake_case(text):
    if text is None:
        return None

    # Remove espaços extras e converte o texto para minúsculo
    text = str(text).strip().lower()

    # Remove acentos e converte caracteres especiais para o equivalente simples
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")

    # Remove caracteres especiais, mantendo apenas letras, números e espaços
    text = re.sub(r"[^\w\s]", "", text)

    # Substitui espaços por underline
    text = re.sub(r"\s+", "_", text)

    return text


# =========================
# 5. TRADUÇÃO DOS NOMES DAS COLUNAS
# =========================

# Dicionário que mapeia os nomes das colunas em português
# para nomes padronizados em inglês
mapeamento_colunas = {
    "Data": "date",
    "Temperatura Máxima": "max_temperature",
    "Temperatura Mínima": "min_temperature",
    "Chuva (mm)": "rain_mm",
    "Código do Clima": "weather_code"
}

# Percorre o dicionário e renomeia as colunas caso elas existam no DataFrame
for coluna_antiga, coluna_nova in mapeamento_colunas.items():
    if coluna_antiga in df.columns:
        df = df.withColumnRenamed(coluna_antiga, coluna_nova)


# =========================
# 6. PADRONIZAÇÃO DOS NOMES DAS COLUNAS
# =========================

# Aplica a função to_snake_case em todos os nomes das colunas
# para garantir padronização estrutural
df = df.toDF(*[to_snake_case(c) for c in df.columns])


# =========================
# 7. PADRONIZAÇÃO DOS VALORES STRING
# =========================

# Percorre todas as colunas e verifica quais são do tipo string
for campo, tipo in df.dtypes:
    if tipo == "string":
        # Remove espaços no início e no fim do conteúdo
        # e transforma o valor em minúsculo
        df = df.withColumn(campo, trim(lower(col(campo))))

        # Substitui espaços internos por underline (_)
        # Exemplo: "Light Rain" -> "light_rain"
        df = df.withColumn(campo, regexp_replace(col(campo), r"\s+", "_"))


# =========================
# 8. TRATAMENTO DE CAMPOS VAZIOS
# =========================

# Percorre todas as colunas do DataFrame
for campo in df.columns:
    df = df.withColumn(
        campo,

        # Se o valor for vazio ("") ou tiver apenas espaços,
        # ele será convertido para null
        when(trim(col(campo)) == "", None).otherwise(col(campo))
    )


# =========================
# 9. CRIAÇÃO DA DESCRIÇÃO DO CLIMA
# =========================

# Verifica se a coluna weather_code existe antes de aplicar a transformação
if "weather_code" in df.columns:
    df = df.withColumn(
        "weather_description",

        # Traduz o código numérico do clima em uma descrição textual padronizada
        # baseada nos códigos oficiais retornados pela API meteorológica
        when(col("weather_code") == "0", "clear_sky")
        .when(col("weather_code") == "1", "mainly_clear")
        .when(col("weather_code") == "2", "partly_cloudy")
        .when(col("weather_code") == "3", "overcast")
        .when(col("weather_code") == "45", "fog")
        .when(col("weather_code") == "48", "depositing_rime_fog")
        .when(col("weather_code") == "51", "light_drizzle")
        .when(col("weather_code") == "53", "moderate_drizzle")
        .when(col("weather_code") == "55", "dense_drizzle")
        .when(col("weather_code") == "56", "light_freezing_drizzle")
        .when(col("weather_code") == "57", "dense_freezing_drizzle")
        .when(col("weather_code") == "61", "light_rain")
        .when(col("weather_code") == "63", "moderate_rain")
        .when(col("weather_code") == "65", "heavy_rain")
        .when(col("weather_code") == "66", "light_freezing_rain")
        .when(col("weather_code") == "67", "heavy_freezing_rain")
        .when(col("weather_code") == "71", "light_snow_fall")
        .when(col("weather_code") == "73", "moderate_snow_fall")
        .when(col("weather_code") == "75", "heavy_snow_fall")
        .when(col("weather_code") == "77", "snow_grains")
        .when(col("weather_code") == "80", "light_rain_showers")
        .when(col("weather_code") == "81", "moderate_rain_showers")
        .when(col("weather_code") == "82", "violent_rain_showers")
        .when(col("weather_code") == "85", "light_snow_showers")
        .when(col("weather_code") == "86", "heavy_snow_showers")
        .when(col("weather_code") == "95", "thunderstorm")
        .when(col("weather_code") == "96", "thunderstorm_with_light_hail")
        .when(col("weather_code") == "99", "thunderstorm_with_heavy_hail")

        # Caso venha algum código não mapeado, o valor será marcado como unknown
        .otherwise("unknown")
    )


# =========================
# 10. REMOÇÃO DE LINHAS DUPLICADAS
# =========================

# Remove registros completamente duplicados do DataFrame
# Isso evita repetição de linhas idênticas no arquivo final
df = df.dropDuplicates()


# =========================
# 11. VALIDAÇÃO VISUAL DO RESULTADO
# =========================

# Exibe a estrutura final do DataFrame
# Mostra nomes de colunas e tipos de dados
df.printSchema()

# Mostra até 10 registros tratados para conferência visual
df.limit(10).show(truncate=False)


# =========================
# 12. CARGA / EXPORTAÇÃO FINAL (L DO ETL)
# =========================

# Converte o DataFrame Spark para Pandas
# O Pandas será usado apenas para facilitar a geração de um único CSV final
df_pandas = df.toPandas()

try:
    # Se o arquivo final já existir, ele será removido antes de salvar o novo
    if os.path.exists(arquivo_saida):
        os.remove(arquivo_saida)

    # Exporta o DataFrame tratado para CSV
    df_pandas.to_csv(arquivo_saida, index=False, encoding="utf-8")

    print(f"Arquivo tratado salvo com sucesso em: {arquivo_saida}")

# Caso o arquivo esteja aberto em outro programa (como Excel),
# será exibida uma mensagem amigável de erro
except PermissionError:
    print("Erro: não foi possível salvar o arquivo. Verifique se o CSV está aberto em outro programa.")

root
 |-- date: string (nullable = true)
 |-- max_temperature: string (nullable = true)
 |-- min_temperature: string (nullable = true)
 |-- rain_mm: string (nullable = true)
 |-- weather_code: string (nullable = true)
 |-- weather_description: string (nullable = false)

+----------+---------------+---------------+-------+------------+----------------------------+
|date      |max_temperature|min_temperature|rain_mm|weather_code|weather_description         |
+----------+---------------+---------------+-------+------------+----------------------------+
|2026-04-08|30.0           |19.5           |1.7    |51          |light_drizzle               |
|2026-04-07|28.2           |19.0           |0.6    |3           |overcast                    |
|2026-04-04|26.6           |19.8           |1.8    |3           |overcast                    |
|2026-04-01|25.4           |18.2           |7.3    |96          |thunderstorm_with_light_hail|
|2026-04-11|24.4           |19.2           |1.3    |51          